# Anomaly Detection Models — Environmental Anomaly Detector

Train Isolation Forest, Local Outlier Factor (LOF), and build an LSTM Autoencoder
with PyTorch for environmental anomaly detection.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

## Modeling Steps

1. Scale features and prepare sequences for LSTM
2. Train Isolation Forest with contamination tuning
3. Train Local Outlier Factor (LOF)
4. Build and train LSTM Autoencoder in PyTorch
5. Compare anomaly scores across all three methods

In [ ]:
# Load data
df = pd.read_parquet('../data/processed/feature_matrix.parquet')
feature_cols = [c for c in df.columns if c.endswith(('_mean', '_std', '_zscore'))]
X = df[feature_cols].dropna().values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
iso_labels = iso.fit_predict(X_scaled)
print(f'Isolation Forest anomalies: {(iso_labels == -1).sum()} / {len(iso_labels)}')

# LOF
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_labels = lof.fit_predict(X_scaled)
print(f'LOF anomalies: {(lof_labels == -1).sum()} / {len(lof_labels)}')

In [ ]:
# LSTM Autoencoder
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, n_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        enc_out, (h, c) = self.encoder(x)
        dec_out, _ = self.decoder(enc_out, (h, c))
        return self.fc(dec_out)

# Prepare sequences
seq_len = 24
sequences = [X_scaled[i:i+seq_len] for i in range(len(X_scaled) - seq_len)]
X_seq = torch.FloatTensor(np.array(sequences))

model = LSTMAutoencoder(input_dim=X_scaled.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Training loop (placeholder)
print(f'LSTM Autoencoder: input_dim={X_scaled.shape[1]}, sequences={len(sequences)}')